# Econ 390 - Problem Set 12
## by INSERT NAME HERE
This problem set will cover Difference in Differences and RegEx. Be sure to import any packages your code needs to run and make sure the code runs without any errors.

1. [1 Point] Write a RegEx to find all Social Security Numbers in a string (it should accommodate `XXX-XX-XXXX` and `XXXXXXXXX`)

2. [2 Points] Write a RegEx to find prices either in `$` or `€` with optional commas or periods separating thousands, millions, etc. and optional comma or period for **two digit** decimals

3. Run the code block below to generate a dataset of 100 "Firms" sales numbers across six years. Assume starting in year 4 a policy went into effect that only effected `firm_type == 1` and we're trying to evaluate the effect of the policy. It's ambiguous as to whether it will improve sales or make them worse.

In [1]:
# Import the random package as we will be randomizing firm sales
import random
import pandas as pd
# Set the seed for reproducability
random.seed(a=390)

# Initialize number of firms, periods, and firm_types
nfirms = 100
nperiods = 6
firm_types = 2

# Create lists for our variables of the appropriate size
firm = [0]*nfirms*nperiods
period = [0]*nfirms*nperiods
after = [0]*nfirms*nperiods
firm_type = [0]*nfirms*nperiods
sales = [0]*nfirms*nperiods
sales_change = [0]*nfirms*nperiods
sales_slope = [0]*firm_types
sales_slope_new = [0]*firm_types

# Generate a random sales slope for each firm type before and after
for i in range(firm_types):
    sales_slope[i] = random.uniform(0.5,1.5)
    sales_slope_new[i] = random.uniform(0.5,1.5)
# Set the new sales slope for firm_type == 0 equal to the old one since they're untreated
sales_slope_new[0] = sales_slope[0]

# Fill out the lists by looping through the firms and periods
index = 0
for i in range(nfirms):
    for j in range(nperiods):
        firm[index] = i
        period[index] = j
        firm_type[index] = i % 2
        # If we're in the before periods, use the old slope
        if j < 3:
            sales[index] = (j+1)*sales_slope[i % 2] + random.gauss(mu=0,sigma=0.05)
            after[index] = 0
        # In the after periods, use the new slope
        else:
            sales[index] = (j+1)*sales_slope_new[i % 2] + random.gauss(mu=0,sigma=0.05)
            after[index] = 1

        # If we are not in period zero, calculate the sales_change
        if j != 0:
            sales_change[index] = sales[index] - sales[index-1]
        else:
            sales_change[index] = None

        index += 1

# Convert to a dataframe
df = pd.DataFrame({"firm":firm,"period":period,"after":after,"firm_type":firm_type,"sales":sales,"sales_change":sales_change})

- [2 Point] Create a scatterplot with matplotlib of sales over time, with the dots being colored by their firm type and a vertical line at 2.5
  - *Hint: To do so with matplotlib alone, create a scatter of just `firm_type == 0` with one color, then add another scatter of `firm_type == 1` with a different color*
  - In the cell below do the same thing but with `sales_change` instead of sales
  - Reflect on how these two compare and if this looks like a good Diff-in-Diff

- [2 Points] Using `statsmodels.formula.api`, run a Difference in Differences regression, call it `model1`, on the slope of sales across firm type and after
  - *Hint 1: The first regression equation is $sales_{i,t} = \alpha + \beta_1 firm\_type_i + \beta_2 after_t + \beta_3 firm\_type_i \times after_t + \beta_4 period_t + \beta_5 firm\_type_i \times period_t + \beta_6 after_t \times period_t +\beta_7 firm\_type_{i} \times after_t \times period_t + \epsilon_{i,t}$ where $i$ is firm and $t$ is time period*
  - *Hint 2: There is a simple way to include all of these individual regressors in the `smf.ols` function*
  - Use heteroskedastic standard errors
  - Print a summary of the results
  - Reflect on the results

- [1 Point] Run another regression, call it `model2`, this time of the change in sales on firm type interacted with after
    - Notice that there is an impact effect of the policy on the change in sales.
    - To account for this, create an indicator variable for being in the first policy year (i.e. `period == 3`) and include that interacted with firm type in the regression
    - *Hint: This regression equation is $sales\_change_{i,t} = \alpha + \beta_1 firm\_type_i + \beta_2 after + \beta_3 firm\_type_i \times after_t + \beta_4 firm\_type_i \times year\_three_t$*
  - Use heteroskedastic standard errors
  - Print a summary of the results
  - Reflect on the results

- [1 Point] Using only the regression results, what is the estimated effect of the policy? Does it matter whether we run the regression on the sales vs. the change in sales? *Hint: What is $\beta_7$ in `model1` and $\beta_3$ in `model2`?*

- [1 Point]  What is the actual effect? *Hint: What is `sales_slope_new - sales_slope` for `firm_type == 1`*
    - Is the real value within the 95% confidence interval for the estimated effect above? Answer in the markdown cell below.

- Notice that if we only look at the levels of sales, it looks like this data violates the "parallel trends" assumption. But, when we look at the change in sales, the two groups look parallel. The assumption is about the *differences* being parallel, so it works! And knowing that the *differences* are parallel, we can still run the regression on the levels (`model1`) and recover the effect of the policy ($\beta_7$).